# 第一部分：数据获取

本 Notebook 完成以下数据下载：
1. **10 只 A 股股票**的后复权日度行情（2020-01-01 至今）
2. **沪深 300 指数** + **中证 500 指数**日度数据
3. **宏观经济指标**：CPI 同比增速（月度）+ M2 同比增速（月度）
4. **财务指标**：ROE、净利润率（近 5 年度），整理为长格式

In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import os
from datetime import datetime
import time

print(f"akshare 版本: {ak.__version__}")
print(f"数据下载开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 创建下载日志
log_file = "download_log.txt"
with open(log_file, "w", encoding="utf-8") as f:
    f.write(f"数据下载日志 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n")

def write_log(msg):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{ts}] {msg}\n"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(entry)
    print(entry.strip())

## 1.1 股票列表

选取 10 只股票，覆盖银行、汽车、能源、白酒、通讯、物流、房地产共 7 个行业
（要求至少 5 个行业，每个行业至多 2 只）。

In [ ]:
stocks = [
    {"code": "000001", "name": "平安银行", "industry": "银行",
     "reason": "银行板块龙头，资产规模大，代表银行业经营水平"},
    {"code": "600036", "name": "招商银行", "industry": "银行",
     "reason": "零售银行标杆，ROE 行业领先，与平安银行形成对比"},
    {"code": "002594", "name": "比亚迪",   "industry": "汽车",
     "reason": "新能源汽车龙头，近年业绩爆发式增长"},
    {"code": "300750", "name": "宁德时代", "industry": "能源",
     "reason": "动力电池全球龙头，新能源产业链核心标的"},
    {"code": "601012", "name": "隆基绿能", "industry": "能源",
     "reason": "光伏行业龙头，与宁德时代形成新能源细分对比"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒",
     "reason": "A 股市值标杆，消费板块代表性极强"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯",
     "reason": "5G 通信设备龙头，受益于数字经济政策"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流",
     "reason": "快递行业龙头，直营模式代表"},
    {"code": "600048", "name": "保利发展", "industry": "房地产",
     "reason": "央企地产龙头，反映房地产行业周期"},
    {"code": "002475", "name": "立讯精密", "industry": "通讯",
     "reason": "苹果产业链核心供应商，消费电子代表"},
]

stocks_df = pd.DataFrame(stocks)
display(stocks_df[["code", "name", "industry", "reason"]])

## 1.2 下载个股日度行情数据

使用 `akshare` 的 `stock_zh_a_hist()` 接口，获取后复权日度行情。
字段要求：日期、开盘价、收盘价、最高价、最低价、成交量、成交额。

In [ ]:
start_date = "20200101"
end_date = datetime.now().strftime("%Y%m%d")

print(f"下载时间范围: {start_date} 至 {end_date}")
print("复权方式: 后复权 (hfq)")
print("数据源: 新浪 (东方财富接口暂不可用)")
print("-" * 60)

os.makedirs("data/stock", exist_ok=True)

for stock in stocks:
    code_ = stock["code"]
    name_ = stock["name"]
    # 深市: 00/30 → sz, 沪市: 60 → sh
    prefix = "sh" if code_.startswith("6") else "sz"
    symbol = f"{prefix}{code_}"
    try:
        df = ak.stock_zh_a_daily(symbol=symbol, adjust="hfq")
        # 筛选时间范围
        df["date"] = pd.to_datetime(df["date"])
        df = df[(df["date"] >= "2020-01-01") & (df["date"] <= end_date)]
        # 统一列名: 保留 date,open,high,low,close,volume,amount(=成交额/turnover)
        cols = ["date", "open", "high", "low", "close", "volume", "amount"]
        df = df[cols].rename(columns={"amount": "turnover"})
        path = f"data/stock/stock_{code_}.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        write_log(f"SUCCESS  stock_{code_} ({name_})  shape={df.shape}")
    except Exception as e:
        write_log(f"FAILED   stock_{code_} ({name_})  Error: {e}")
    time.sleep(0.5)

print("-" * 60)
print("个股数据下载完成！")

In [ ]:
# 检查下载结果
sample = pd.read_csv("data/stock/stock_000001.csv")
print(f"样例数据（平安银行）：shape={sample.shape}")
print(f"列名: {sample.columns.tolist()}")
display(sample.head())

## 1.3 下载市场指数数据

- **沪深 300**（000300）：作为 CAPM 分析的市场基准（必选）
- **中证 500**（000905）：代表中小市值公司，与沪深 300 形成互补（自选）

**选择中证 500 的理由**：中证 500 代表中小市值公司，与沪深 300（大盘蓝筹）形成互补，可以更全面地反映市场结构。

In [5]:
os.makedirs("data/index", exist_ok=True)

# 沪深 300
print("下载沪深 300...")
try:
    hs300 = ak.stock_zh_index_daily(symbol="sh000300")
    hs300["date"] = pd.to_datetime(hs300["date"])
    hs300 = hs300[(hs300["date"] >= "2020-01-01") & (hs300["date"] <= end_date)]
    col_map = {"open": "idx_open", "close": "idx_close", "high": "idx_high",
               "low": "idx_low", "volume": "idx_volume"}
    hs300 = hs300.rename(columns={k: v for k, v in col_map.items() if k in hs300.columns})
    hs300.to_csv("data/index/index_000300.csv", index=False, encoding="utf-8-sig")
    write_log(f"SUCCESS  index_000300 (沪深300)  shape={hs300.shape}")
except Exception as e:
    write_log(f"FAILED   index_000300  Error: {e}")

# 中证 500
print("下载中证 500...")
try:
    zz500 = ak.stock_zh_index_daily(symbol="sh000905")
    zz500["date"] = pd.to_datetime(zz500["date"])
    zz500 = zz500[(zz500["date"] >= "2020-01-01") & (zz500["date"] <= end_date)]
    col_map = {"open": "idx_open", "close": "idx_close", "high": "idx_high",
               "low": "idx_low", "volume": "idx_volume"}
    zz500 = zz500.rename(columns={k: v for k, v in col_map.items() if k in zz500.columns})
    zz500.to_csv("data/index/index_000905.csv", index=False, encoding="utf-8-sig")
    write_log(f"SUCCESS  index_000905 (中证500)  shape={zz500.shape}")
except Exception as e:
    write_log(f"FAILED   index_000905  Error: {e}")

下载沪深 300...
[2026-05-23 13:53:50] SUCCESS  index_000300 (沪深300)  shape=(1545, 6)
下载中证 500...
[2026-05-23 13:53:51] SUCCESS  index_000905 (中证500)  shape=(1545, 6)


## 1.4 下载宏观经济指标

- **CPI 同比增速**（必选）：反映居民消费价格变化，是央行货币政策的重要参考
- **M2 同比增速**（自选）：反映货币供应量，与股市流动性密切相关

**选择 M2 的理由**：M2 同比增速反映市场流动性水平。货币宽松（M2 高增）通常利好股市估值，货币紧缩则可能抑制股市表现。选择 M2 可以探讨流动性对股票市场的影响。

> 注：原始接口  已不存在，改用 。

In [ ]:
os.makedirs("data/macro", exist_ok=True)

# CPI 月度同比增速
print("下载 CPI 月度同比增速...")
try:
    cpi_df = ak.macro_china_cpi_monthly()
    print(f"CPI 原始列名: {cpi_df.columns.tolist()}")
    print(f"CPI 数据量: {len(cpi_df)}")
    display(cpi_df.head())
    cpi_df.to_csv("data/macro/macro_cpi.csv", index=False, encoding="utf-8-sig")
    write_log(f"SUCCESS  macro_cpi  shape={cpi_df.shape}")
except Exception as e:
    write_log(f"FAILED   macro_cpi  Error: {e}")

In [ ]:
# M2 月度同比增速（使用 macro_china_money_supply）
print("下载 M2 月度同比增速...")
try:
    m2_raw = ak.macro_china_money_supply()
    print(f"M2 原始列名: {m2_raw.columns.tolist()}")
    print(f"M2 数据量: {len(m2_raw)}")
    # 提取需要的列
    m2_df = m2_raw[["月份", "货币和准货币(M2)-同比增长"]].copy()
    m2_df = m2_df.rename(columns={
        "月份": "month",
        "货币和准货币(M2)-同比增长": "m2_yoy"
    })
    # 清洗: 移除空值和文本月份，提取 2020 年后的数据
    m2_df = m2_df.dropna(subset=["m2_yoy"])
    m2_df["year_month"] = m2_df["month"].str.extract(r"(\d{4})年(\d{2})月")
    m2_df = m2_df.dropna(subset=["year_month"]).copy()
    # 暂时保留完整数据，后续清洗时再筛选
    print(f"M2 清洗后数据量: {len(m2_df)}")
    display(m2_df.head())
    m2_df.to_csv("data/macro/macro_m2.csv", index=False, encoding="utf-8-sig")
    write_log(f"SUCCESS  macro_m2  shape={m2_df.shape}")
except Exception as e:
    write_log(f"FAILED   macro_m2  Error: {e}")

## 1.5 下载财务指标

获取 10 只股票近 5 个年度的 **ROE（净资产收益率）** 和 **净利润率**，整理为长格式：

`code, year, indicator, value`

In [ ]:
os.makedirs("data/finance", exist_ok=True)

finance_rows = []

for stock in stocks:
    code_ = stock["code"]
    name_ = stock["name"]
    try:
        df = ak.stock_financial_abstract(symbol=code_)
        print(f"{name_}({code_}): shape={df.shape}, indicators={len(df)}")
        finance_rows.append({"code": code_, "name": name_, "raw": df})
        write_log(f"SUCCESS  finance_{code_} ({name_})  shape={df.shape}")
    except Exception as e:
        write_log(f"FAILED   finance_{code_} ({name_})  Error: {e}")
    time.sleep(0.5)

In [ ]:
# 整理财务数据为长格式 (code, year, indicator, value)
# stock_financial_abstract 结构: 指标列 + 多个日期列(如20251231)

finance_long_list = []

for item in finance_rows:
    code_ = item["code"]
    name_ = item["name"]
    df = item["raw"]
    
    # 找 ROE 和 销售净利率 行
    roe_rows = df[df["指标"] == "净资产收益率(ROE)"]
    npm_rows = df[df["指标"] == "销售净利率"]
    
    if roe_rows.empty and npm_rows.empty:
        print(f"警告: {name_} 未找到 ROE/净利率指标")
        continue
    
    # 日期列（如 20201231, 20211231...）
    date_cols = [c for c in df.columns if c not in ["选项", "指标"] and str(c).isdigit() and len(str(c)) == 8]
    
    for col in date_cols:
        year = int(str(col)[:4])
        if year < 2020:
            continue
        
        if not roe_rows.empty:
            val = pd.to_numeric(roe_rows.iloc[0][col], errors="coerce")
            if not pd.isna(val):
                finance_long_list.append({
                    "code": code_, "name": name_, "year": year,
                    "indicator": "ROE", "value": float(val)
                })
        if not npm_rows.empty:
            val = pd.to_numeric(npm_rows.iloc[0][col], errors="coerce")
            if not pd.isna(val):
                finance_long_list.append({
                    "code": code_, "name": name_, "year": year,
                    "indicator": "net_profit_margin", "value": float(val)
                })

finance_long_df = pd.DataFrame(finance_long_list)
finance_long_df.to_csv("data/finance/finance_ratios.csv", index=False, encoding="utf-8-sig")
print()
print(f"财务长格式数据: shape={finance_long_df.shape}")
display(finance_long_df.head(10))
write_log(f"SUCCESS  finance_ratios_long  shape={finance_long_df.shape}")

## 1.6 检查下载结果

In [ ]:
# 显示下载日志
print("=" * 60)
with open("download_log.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
print(f"数据下载完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n请继续运行 02_clean.ipynb 进行数据清洗")